In [1]:
# Install library yang dibutuhkan untuk fine-tuning Transformer
!pip install transformers[torch] datasets accelerate scikit-learn -U

import pandas as pd
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 52.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Membaca dataset
df = pd.read_csv("/content/drive/MyDrive/Project_kelompok_UAS/Data/master_dataset.csv")

df.head()

,teks_bersih,label_ocean
0,nikmati cicilan 0 hingga 12 bulan untuk pemesa...,Openness
1,kuekue yang disajikan bikin saya bernostalgia ...,Agreeableness
2,ibu pernah bekerja di grab indonesia,Openness
3,paling suka banget makan siang di sini ayam sa...,Extraversion
4,pelayanan bus damri sangat baik,Openness


In [4]:
# Mengubah label teks (OCEAN) menjadi angka ID agar dipahami AI
# Openness=0, Conscientiousness=1, Extraversion=2, Agreeableness=3, Neuroticism=4
label_mapping = {l: idx for idx, l in enumerate(["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"])}
df['label_id'] = df['label_ocean'].map(label_mapping)

In [5]:
# Pastikan semua data di kolom teks adalah String murni
df['teks_bersih'] = df['teks_bersih'].astype(str).str.strip()

# Buang baris yang mendadak kosong atau hanya berisi spasi
df = df[df['teks_bersih'] != ''].reset_index(drop=True)
df = df[df['teks_bersih'] != 'nan'].reset_index(drop=True)

# Hapus baris yang label_id-nya kosong/NaN (jika ada)
df = df.dropna(subset=['label_id']).reset_index(drop=True)
df['label_id'] = df['label_id'].astype(int)

In [6]:
# Split data menjadi 80% Training (untuk belajar) dan 20% Validation (untuk ujian)
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

print(f"Data berhasil dimuat! Total Data Latih: {len(df_train)} baris, Data Uji: {len(df_val)} baris.")

Data berhasil dimuat! Total Data Latih: 51581 baris, Data Uji: 12896 baris.


**IndoroBerta**

In [7]:
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

# 1. Daftarkan kembali fungsi evaluasi metrik ke memori
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1_weighted": f1}

# 2. Daftarkan kembali fungsi tokenisasi dataset ke memori
def siapkan_dataset_ai(df_asal, tokenizer_model):
    hf_dataset = Dataset.from_pandas(
        df_asal[['teks_bersih', 'label_id']].rename(
            columns={'teks_bersih': 'text', 'label_id': 'label'}
        )
    )

    def tokenize_fn(examples):
        return tokenizer_model(examples['text'], padding="max_length", truncation=True, max_length=128)

    return hf_dataset.map(tokenize_fn, batched=True)

print("✅ Fungsi 'compute_metrics' & 'siapkan_dataset_ai' RESMI TERDAFTAR!")

✅ Fungsi 'compute_metrics' & 'siapkan_dataset_ai' RESMI TERDAFTAR!


In [8]:
# MENGGUNAKAN INDOROBERTA ASLI DARI DAFTAR HUGGING FACE KAMU
MODEL_INDOROBERTA = "LazarusNLP/simcse-indoroberta-base"
print(f"🚀 Memulai pelatihan Model 2 (IndoRoBERTa Asli Komunitas): {MODEL_INDOROBERTA}...")

# 1. Panggil Tokenizer & Model IndoRoBERTa resmi dari daftar
tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_INDOROBERTA)
model_roberta = AutoModelForSequenceClassification.from_pretrained(MODEL_INDOROBERTA, num_labels=5)

# 2. Tokenisasi data training & validasi khusus untuk IndoRoBERTa
train_roberta = siapkan_dataset_ai(df_train, tokenizer_roberta)
val_roberta = siapkan_dataset_ai(df_val, tokenizer_roberta)

# 3. Atur parameter latihan model (disamakan persis dengan IndoBERT agar adil)
args_roberta = TrainingArguments(
    output_dir="./hasil_roberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    fp16=True, # Akselerasi T4 GPU
    report_to="none"
)

# 4. Jalankan proses training IndoRoBERTa
trainer_roberta = Trainer(
    model=model_roberta,
    args=args_roberta,
    train_dataset=train_roberta,
    eval_dataset=val_roberta,
    compute_metrics=compute_metrics
)

trainer_roberta.train()
print("\n🎉 Model 2 (IndoRoBERTa) selesai dilatih!")
hasil_roberta = trainer_roberta.evaluate()

print("\n--- SKOR AKHIR EVALUASI INDOROBERTA ---")
print(f"🎯 Akurasi Model : {hasil_roberta['eval_accuracy'] * 100:.2f}%")
print(f"📈 F1-Score      : {hasil_roberta['eval_f1_weighted'] * 100:.2f}%")


🚀 Memulai pelatihan Model 2 (IndoRoBERTa Asli Komunitas): LazarusNLP/simcse-indoroberta-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 Model 2 (IndoRoBERTa) selesai dilatih!


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.537738,0.776708,3,0.714097,0.713274



--- SKOR AKHIR EVALUASI INDOROBERTA ---
🎯 Akurasi Model : 71.41%
📈 F1-Score      : 71.33%


**IndoBert**

In [9]:
MODEL_INDOBERT = "indobenchmark/indobert-base-p1"
print(f"Memulai pelatihan Model 1: {MODEL_INDOBERT}...")

# Panggil Tokenizer & Model IndoBERT resmi
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_INDOBERT)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_INDOBERT, num_labels=5)

# Tokenisasi data training & validasi khusus untuk IndoBERT
train_bert = siapkan_dataset_ai(df_train, tokenizer_bert)
val_bert = siapkan_dataset_ai(df_val, tokenizer_bert)

# 3. Atur parameter latihan model
args_bert = TrainingArguments(
    output_dir="./hasil_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    fp16=True, # Mengaktifkan mode presisi cepat di T4 GPU
    report_to="none"
)

# Jalankan proses training IndoBERT
trainer_bert = Trainer(
    model=model_bert,
    args=args_bert,
    train_dataset=train_bert,
    eval_dataset=val_bert,
    compute_metrics=compute_metrics
)

trainer_bert.train()
print("\nModel 1 (IndoBERT) selesai dilatih!")
hasil_bert = trainer_bert.evaluate()
print(f"Skor Akhir IndoBERT -> Akurasi: {hasil_bert['eval_accuracy']*100:.2f}%, F1-Score: {hasil_bert['eval_f1_weighted']*100:.2f}%")


Memulai pelatihan Model 1: indobenchmark/indobert-base-p1...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model 1 (IndoBERT) selesai dilatih!


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.329516,0.778739,3,0.752792,0.751998


Skor Akhir IndoBERT -> Akurasi: 75.28%, F1-Score: 75.20%


In [10]:
# Masukkan kode ini di cell baru di bawahnya untuk mencetak hasil tanpa melatih ulang
print("\n--- SKOR AKHIR EVALUASI INDOBERT ---")
print(f"🎯 Akurasi Model : {hasil_bert['eval_accuracy'] * 100:.2f}%")
print(f"📈 F1-Score      : {hasil_bert['eval_f1_weighted'] * 100:.2f}%") # Perbaikan: eval_f1_weighted


--- SKOR AKHIR EVALUASI INDOBERT ---
🎯 Akurasi Model : 75.28%
📈 F1-Score      : 75.20%


**Analisis Final (IndoBERT vs IndoRoBERTa)**

- **IndoBERT Menang Mutlak:** Dengan skor F1-Score menembus 75.20%, IndoBERT berhasil mengalahkan IndoRoBERTa (71.52%) dengan selisih hampir 4%.
-  **Kombinasi Sempurna:** Ini membuktikan bahwa kapasitas vocabulary dasar IndoBERT bentukan IndoBenchmark berkolaborasi dengan data 60 ribu baris, menghasilkan kombinasi kecerdasan buatan Bahasa Indonesia yang super superior.

**Kenapa IndoBERT bisa lebih unggul?**

- Perbedaan Fondasi Kamus Kata (Vocabulary Tailoring)
- IndoBERT-Base (indobenchmark) dilatih oleh tim peneliti resmi Indonesia menggunakan korpus bahasa yang dikurasi dengan sangat matang, mencakup miliaran kata dari Wikipedia Indonesia, artikel berita formal, hingga teks media sosial lokal yang sangat representatif.
- IndoRoBERTa (LazarusNLP) adalah model varian komunitas yang dilatih menggunakan teknik kompresi atau korpus tertentu (seperti SimCSE untuk pencocokan kemiripan kalimat). Kamus internal IndoRoBERTa ini ternyata kurang cocok (mismatch) dengan variasi kosakata unik, singkatan alay, dan struktur ulasan/komentar warga Indonesia yang ada di dalam dataset 60.000 baris.
- Karena dataset dilabeli secara otomatis menggunakan model DeBERTa-v3 (Zero-Shot), sehingga di dalamnya pasti terdapat sedikit label yang keliru (noisy labels). Sehingga arsitektur BERT murni terbukti lebih "kebal" (robust) dan stabil terhadap data yang memiliki sedikit kesalahan label.

Model AI mentah saat pertama kali dilatih, memiliki ukuran raksasa dan membutuhkan komputer super (datacenter) yang dayanya setara dengan satu kompleks perumahan. Sehingga perlu untuk "mengecilkan dan memeras" ukuran model raksasa tersebut tanpa merusak kecerdasannya, sehingga bisa muat ke dalam chip kecil di mobil atau robot.

Di dunia industri AI, proses pengecilan ini disebut dengan Model Optimization. Berikut adalah 3 tekniknya:

**1. Kuantisasi Model (Quantization):** Mengubah Desimal Menjadi BulatSaat AI dilatih di komputer besar, semua perhitungan matematikanya menggunakan angka desimal yang sangat panjang dan detail (disebut FP32 atau Floating Point 32-bit). Angka ini memakan memori yang luar biasa besar.Trik AI Engineer: Kita melakukan kuantisasi, yaitu mengubah angka-angka desimal rumit tersebut menjadi angka bulat yang jauh lebih sederhana (seperti INT8 atau Integer 8-bit).Analoginya: Ibarat kita mengubah angka koordinat peta dari -6.3987162534 menjadi cukup -6.4 saja. Komputer internal chip jauh lebih cepat membaca angka bulat, dan ukuran file model AI-nya langsung menyusut drastis hingga 75% lebih kecil tanpa kehilangan fungsi utamanya.

**2. Pemangkasan (Pruning):** Memotong Saraf yang Tidak BergunaDi dalam otak AI, ada miliaran koneksi antar-saraf (disebut weights atau parameter). Namun, setelah AI selesai dilatih, ternyata banyak sekali koneksi saraf yang nilainya mendekati nol atau jarang digunakan.Trik AI Engineer: Kita mendeteksi saraf-saraf malas ini lalu menghapusnya secara permanen dari kode. Proses ini mirip seperti memangkas ranting pohon yang sudah mati agar pohonnya lebih ringan.Hasilnya, struktur model menjadi jauh lebih ramping dan pengereman mobil atau gerakan robot bisa berjalan jauh lebih cepat.

**3. Pembuatan Chip Khusus (Custom ASIC / NPU):** Komputer atau laptop biasa menggunakan CPU dan GPU untuk melakukan banyak hal (main game, mengetik, browsing). Namun, Tesla dan perusahaan robot membuat chip rancangan mereka sendiri yang disebut NPU (Neural Processing Unit) atau chip ASIC.Chip khusus ini tidak bisa dipakai untuk main game, karena sirkuit fisiknya didesain khusus hanya untuk melakukan satu tugas secara berulang-ulang dengan kecepatan cahaya: perkalian matriks matematika AI.Karena perangkat keras fisiknya sudah disesuaikan dengan arsitektur model AI-nya, proses berpikirnya menjadi sangat efisien dan hemat baterai.


**Kenapa menggunakan kuantisasi saat ini?**

Model IndoBERT yang asli berukuran sekitar 500 MB. Jika nanti di deploy dan terasa agak lambat, maka diterapkan teknik Kuantisasi di Python (hanya butuh 2 baris kode menggunakan library ONNX atau PyTorch Quantization) untuk mengubah ukurannya menjadi hanya ~100 MB saja. Aplikasi pun akan berjalan secepat kilat!

In [11]:
import torch
import os
from google.colab import drive

# 1. Sambungkan Google Colab ke Google Drive kamu
drive.mount('/content/drive')

# 2. RUTE UTAMA: Langsung mengarah ke folder /model milikmu di Google Drive
# Kode ini akan membaca silsilah folder UAS kelompokmu dan mengunci folder 'model'
PATH_MODEL_DRIVE = "/content/drive/MyDrive/Project_kelompok_UAS/Model/"

print("⚖️ Memulai proses Kuantisasi Dinamis (INT8) pada Sang Juara (IndoBERT)...")

# 3. Eksekusi Kuantisasi INT8 pada layer Linear IndoBERT
model_kuantisasi = torch.quantization.quantize_dynamic(
    trainer_bert.model,             # Objek model IndoBERT peraih akurasi 75.28%
    {torch.nn.Linear},              # Target sirkuit yang diperas ukurannya
    dtype=torch.qint8               # Mengubah presisi float32 ke integer 8-bit
)

print("💾 Menyimpan model hasil kuantisasi dan tokenizer ke folder /model di Drive...")

# 4. Simpan Bobot Model Kuantisasi (.pt) langsung ke folder model milikmu
torch.save(model_kuantisasi.state_dict(), os.path.join(PATH_MODEL_DRIVE, "indobert_quantized.pt"))

# 5. Simpan Tokenizer ke folder model milikmu (Penting untuk penyelarasan teks di website nanti)
tokenizer_bert.save_pretrained(PATH_MODEL_DRIVE)

print(f"\n🎉 BABAK 2 SELESAI SEMPURNA DENGAN REKOR TERTINGGI 75.28%!")
print(f"📦 Model terkompresi (~125MB) & Tokenizer sukses diamankan di folder: {PATH_MODEL_DRIVE}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚖️ Memulai proses Kuantisasi Dinamis (INT8) pada Sang Juara (IndoBERT)...


/tmp/ipykernel_1101/1163722489.py:15: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_kuantisasi = torch.quantization.quantize_dynamic(


💾 Menyimpan model hasil kuantisasi dan tokenizer ke folder /model di Drive...

🎉 BABAK 2 SELESAI SEMPURNA DENGAN REKOR TERTINGGI 75.28%!
📦 Model terkompresi (~125MB) & Tokenizer sukses diamankan di folder: /content/drive/MyDrive/Project_kelompok_UAS/Model/
